<a href="https://colab.research.google.com/github/elhamod/IS834_Fall_2026/blob/main/Session%2006%20-%20Visualization%20and%20Exploratory%20Data%20Analysis/06_Groupby_and_Data_Viz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS834 — Session 6

Work through this notebook in Colab. The lecture cells are filled in and ready to run; the
**Warmups** and **Exercises** are yours to complete — write your code in the `# Your code here` cells and your
reasoning in the `*Provide your answer here*` cells.

Every dataset is read straight from the course repository, so there is nothing to upload or install.

*Adapted from Brock Tibert's IS834 Fall 2025 session 6. Edited for Fall 2026 by Mohannad Elhamod.*

# Learning Goals

Session 5 turned a table into group summaries. Today we draw them, and learn to tell a chart that
informs from one that misleads.

- Bar and count plots, and ordering categories so the axis means something
- Scatterplots, overplotting, and correlation, including what correlation cannot tell you
- Line plots: when a line is honest, and choosing the time grain
- Distributions: histograms, bin choices, and boxplots as outlier detectors
- Styling a chart for someone else: titles, labels, reference lines, figure size
- Matplotlib and seaborn, and what each one is for

# Setup


In [ ]:
# The three packages this session needs. Colab already has all of them installed.
import pandas as pd
import seaborn as sns              # today's new one: high-level statistical plots
import matplotlib.pyplot as plt    # the lower-level library seaborn is built on

# Every data file this notebook uses is read straight from the course repository.
# There is nothing to upload: DATA is the folder those files live in.
DATA = "https://raw.githubusercontent.com/elhamod/IS834_Fall_2026/main/data/"


> **Why two plotting libraries?** Seaborn draws the plot in one line; matplotlib is what you reach for afterwards to change the title, the labels, or the figure size. Almost every plot in this notebook is one `sns.` call followed by a few `plt.` calls.


In [ ]:
# Load the two datasets we start the session with.
# Both files are read straight from the course repository -- nothing to upload.
survey = pd.read_csv(DATA + "sample-survey-data.csv")
diamonds = pd.read_csv(DATA + "diamonds.csv")

print(survey.shape, diamonds.shape)


# Quick Warmup Exercise

Four questions on the sales workbook, recapping sessions 3 and 4.

**Warmup 1.** Read in the Orders worksheet from the sales workbook, then run your standard review of the dataset.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Warmup 2.** How many rows in the dataset have `Express Air` as the Ship Mode?

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Warmup 3.** Calculate a column called `profit_margin` from the sales and profit columns.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Warmup 4.** Fill in the missing rows of `Product Base Margin` with the average of the column.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## Tidy the column names

The plots below refer to the orders columns by short lowercase names, so we rename them once, here.

In [ ]:
# A quick tip: lowercase the column names and swap spaces for underscores.
orders.columns = orders.columns.str.replace(" ", "_").str.lower()
orders.columns.tolist()


> Why bother? Because `orders.ship_mode` works and `orders.Ship Mode` cannot — a name with a space can only be reached with brackets and quotes. Renaming once at the top saves you from remembering which style each column needs.
>
> Run this cell **once**. Running it twice is harmless here, but a rename cell that is not idempotent is a classic source of "it worked yesterday" notebook bugs.


# Data Visualization

![](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTX2ME0XXdeHQgZJq83JeczsbjnnTQZxbI_kQ&s)

### Resources:

- https://python-graph-gallery.com/
- https://seaborn.pydata.org/index.html
- [Link to All charts](https://python-graph-gallery.com/all-charts/)
- [Seaborn example gallery](https://seaborn.pydata.org/examples/index.html)
- [Animation](https://python-graph-gallery.com/animation/)
- [plotly for interactivity](https://python-graph-gallery.com/plotly/)
- [Plotnine: Based on the Grammar of Graphics](https://plotnine.org/)

### Data Visualization Considerations

-[Financial Times Visual Vocabulary](https://ft-interactive.github.io/visual-vocabulary/)

### Some Notes

- Matplotlib is the foundation for plotting in python.  The defaults are not great, but we can control all aspects of our plot, typically via `plt.`
- Libraries like seaborn put a layer on top of matplotlib and pandas to make nicer plots
- plotnine is a new(er) library that aims to bring the grammar of graphics to python
- While the libraries help us construct plots, we have input the data in the correct shape for the plot to render as we expect. This might mean we have to create aggregations prior to plotting!


In [ ]:
# The dataset most of the plots below use.
diamonds.head(3)


## Bar/Count Plots

These plots let us look at the computed values over categories.


In [ ]:
# A count plot counts the rows in each category for you -- no groupby needed first.
sns.countplot(data=diamonds, x="cut")
plt.show()


> Note the ordering on the x-axis: seaborn used the order the values happen to appear in the file. `Fair` sits between `Ideal` and `Very Good`, which is meaningless. Cut is an **ordinal** variable — the categories have a real low-to-high order — and we have to tell pandas that.

![](https://media.geeksforgeeks.org/wp-content/uploads/20230918125039/2.png)

![](https://media.geeksforgeeks.org/wp-content/uploads/20230918125108/3.png)


In [ ]:
# Declare cut as an ordered category, worst to best. Every plot afterwards inherits that order.
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
diamonds['cut'] = pd.Categorical(diamonds['cut'], categories=cut_order, ordered=True)

sns.countplot(data=diamonds, x="cut")
plt.show()


**This is a data-preparation step, not a plotting step.** Once `cut` is an ordered category, sorting, grouping and plotting all respect the order, and comparisons like `diamonds['cut'] >= 'Very Good'` start working.

The trap: any value not listed in `categories` becomes `NaN` silently. If you typo one level, those rows vanish from every count you run afterwards without an error.


In [ ]:
# hue splits each bar by a second category.
sns.countplot(data=diamonds, x="cut", hue="color")
plt.show()


> Seven colours times five cuts is 35 bars, and reading any single value off it is hard work. `hue` is powerful and easy to overdo — if the reader has to hunt in the legend, use a table instead.


In [ ]:
# For a plot of a computed value, aggregate first, then plot the aggregate.
avg_price = diamonds.groupby("cut", as_index=False)['price'].mean()
print(avg_price)

sns.barplot(data=avg_price, x="cut", y="price")
plt.show()


**Aggregate, then plot** — that is the flow to build a habit around. It also means you can *read the numbers* before you draw them, so a nonsense chart gets caught at the table stage. Note the `as_index=False` from session 5: seaborn wants a tidy DataFrame with ordinary columns, so a grouped result whose labels are stuck in the index has to be `.reset_index()`-ed before it will plot.

And here is the same confounder as session 5's cut-and-colour exercise, now impossible to miss: `Ideal` — the *best* cut — has the **lowest** average price (about \$3,458), while `Fair`, the worst, comes second highest (\$4,359). Better-cut stones in this dataset tend to be smaller, and carat dominates price far more than cut does.

In [ ]:
# A reference line makes a bar chart comparative instead of just descriptive.
sns.barplot(data=avg_price, x="cut", y="price")
plt.axhline(diamonds.price.mean(), color="red", linestyle="--")
plt.title("Average price by cut, versus the overall average")
plt.ylabel("Average price (USD)")
plt.xlabel("Cut")
plt.show()


> `plt.axhline` draws a horizontal line at whatever y-value you give it — here the overall mean price. Any matplotlib command you run after a seaborn call lands on the same plot, which is how titles, labels and annotations get added.


## Scatterplots and Correlation

Scatterplots let us compare two numeric variables (x and y are numeric) and help us visualize the relationship between these two variables.  Optionally we can overlay data onto the plot.


In [ ]:
# Two numeric columns, one point per row.
sns.scatterplot(data=diamonds, x="carat", y="price")
plt.show()


> 54,000 points on one panel is *overplotting*: the dark mass in the lower left could be a hundred diamonds or ten thousand and you cannot tell. Two standard fixes come next — make the points transparent, and drop the white outline each marker carries by default.


In [ ]:
# hue shades the points by a category.
sns.scatterplot(data=diamonds, x="carat", y="price", hue="cut")
plt.show()


In [ ]:
# alpha makes points transparent so density becomes visible; edgecolor=None removes marker outlines.
sns.scatterplot(data=diamonds, x="carat", y="price", hue="cut", alpha=.3, edgecolor=None)

plt.title("My First Scatterplot")
plt.xlabel("Carat")
plt.ylabel("Diamond Price")

plt.show()

Now the shape is readable: price rises with carat, the spread widens as stones get bigger, and there are vertical stacks at the round carat weights (0.5, 1.0, 1.5) where cutters stop to hit a marketing threshold.

Two habits worth taking from this cell. Every plot you show someone else gets a title and axis labels — `price` is not a label, "Diamond Price (USD)" is. And ending with `plt.show()` keeps the stray `<Axes: ...>` line out of your output.

### Correlation

Going back to your stats class, we know that we can calculate correlation between numeric variables.  Correlation is a measure how much two variables move together and ranges from -1 (perfectly negative correlation) to 1 (perfect positive correlation), and 0 meaning no correlation.

![](https://www.scribbr.de/wp-content/uploads/2021/08/01-correlation-types-1024x415.png)

- We need to evaluate numeric columns
- pandas provides built-in support for this task!


In [ ]:
# select_dtypes pulls out just the numeric columns -- correlation is undefined for text.
diamonds_numeric = diamonds.select_dtypes("number")
diamonds_numeric.info()


> Look at what came along: `rownames` is a row identifier from the original R dataset, not a measurement. It is numeric, so `select_dtypes` grabbed it, and it will happily produce a correlation with price that means nothing. **Drop identifiers before you correlate.**


In [ ]:
# Keep only the real measurements -- the row identifier is not one.
diamonds_numeric = diamonds_numeric.drop(columns="rownames")
diamonds_numeric.corr().round(2)


In [ ]:
# Two columns at a time, when the whole matrix is more than you need.
diamonds['price'].corr(diamonds['carat'])


In [ ]:
# A heatmap turns the matrix into something you can scan. annot=True prints the numbers in the cells.
sns.heatmap(diamonds_numeric.corr(), cmap="Blues", annot=True, fmt=".2f")
plt.title("Correlation between the numeric diamond measurements")
plt.show()


**Read it carefully.** Price and carat sit at 0.92, and `x`, `y` and `z` correlate with carat at 0.95 and above — unsurprising, since those are the stone's physical dimensions and carat is its weight. They are four measurements of the same underlying thing: size.

Two cautions. Correlation only measures a **straight-line** relationship: `depth` shows about -0.01 against price, which means no *linear* trend, not "no relationship". And a colour scale running from white to dark blue makes -1 and 0 look similar; for correlations a diverging palette centred on zero (`cmap="vlag"`) is the honest choice.


# Follow up Questions and Exercises — Part 1: scatterplots and correlation

Work through the following on your own. We will go over the answers together before the end of class.

**Exercise 1.** Read in the baseball dataset from the class website as `mlb`, and perform your quick review of the dataset.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 2.** Keep only the years 2021 to 2023 (`yearID`) — the last three years in the dataset — and save the filtered data as a new DataFrame.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 3.** Create a scatterplot of runs (`R`) on the x-axis and wins (`W`) on the y-axis, using the filtered dataset.

> **Bonus:** label the axes and give it a title. **Double bonus:** shade each point by league (`lgID`).

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 4.** Calculate the correlation between the numeric variables and plot the correlation matrix as a heatmap.

> **Tip:** you may want to change the figure size.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## Line Plots

Lineplots let us look at trends.  Often we would use datetimes to visualize those trends over time.


In [ ]:
# seaborn can aggregate for you: estimator="mean" averages price within each cut before drawing.
sns.lineplot(data=diamonds, x="cut", y="price", estimator="mean")
plt.title("A line plot doing the wrong job")
plt.show()


**This plot runs, and it is still a bad chart.** A line implies that the space *between* two x-values means something — that you could stand halfway between `Good` and `Very Good`. Cut is a set of five labels, so the slopes are decoration.

Keep lines for quantities that genuinely flow: time, above all. The shaded band, by the way, is a 95% confidence interval seaborn adds whenever it aggregates for you — which is another reason to aggregate yourself, so you know exactly what is being drawn.


In [ ]:
# The honest use of a line: total sales per day, aggregated first.
daily_sales = orders.groupby("order_date", as_index=False)['sales'].sum()

plt.figure(figsize=(10, 4))
sns.lineplot(data=daily_sales, x="order_date", y="sales")
plt.title("Daily sales, 2010-2013")
plt.xlabel("Order date")
plt.ylabel("Total sales (USD)")
plt.show()


> Technically correct and practically useless: 1,400 daily points on a narrow chart is a hairball. Daily data is rarely the right grain for a trend — the noise is bigger than the signal you are looking for.


In [ ]:
# Label every order with its month, then total the sales per month.
orders['month'] = orders.order_date.dt.to_period("M").dt.to_timestamp()
monthly_sales = orders.groupby("month", as_index=False)['sales'].sum()

plt.figure(figsize=(10, 4))
sns.lineplot(data=monthly_sales, x="month", y="sales")
plt.title("Monthly sales, 2010-2013")
plt.xlabel("Month")
plt.ylabel("Total sales (USD)")
plt.show()

Same data, readable chart. `.dt.to_period("M")` stamps every order with its month; `.dt.to_timestamp()` turns that back into a real date so the x-axis spaces the months correctly rather than treating them as labels.

The choice of grain — day, week, month, quarter — is an analyst's decision, and it changes what the reader sees. Roll up too far and a real seasonal pattern disappears; too little and noise hides it. State the grain in your title so nobody has to guess.

## Distributions: Histogram and Boxplots

We likely want to better understand the distribution and shape of our data.

#### Histograms

![](https://mokkupai.s3.ap-south-1.amazonaws.com/files/assets/blogs/resources/bcf36a42-da31-435e-86e2-8b8fb550392c.png)

#### Boxplots

![](https://builtin.com/sites/www.builtin.com/files/styles/ckeditor_optimize/public/inline-images/1_boxplots.jpg)


In [ ]:
# A histogram slices one numeric column into bins and counts the rows in each.
sns.histplot(data=diamonds, x="price")
plt.title("Distribution of diamond prices")
plt.show()


> Strongly right-skewed: most diamonds are cheap, a long thin tail runs out past \$18,000. On a shape like this the **mean is not the typical value** — the mean price is about \$3,933 and the median about \$2,401. Quote the median when a distribution has a tail like this.


In [ ]:
# The bin count is a choice, and it changes the story. Same column, three settings.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

sns.histplot(data=diamonds, x="price", bins=5, ax=axes[0])
axes[0].set_title("bins = 5")

sns.histplot(data=diamonds, x="price", bins=30, ax=axes[1])
axes[1].set_title("bins = 30")

sns.histplot(data=diamonds, x="price", bins=100, ax=axes[2])
axes[2].set_title("bins = 100")

plt.tight_layout()
plt.show()

Five bins hide everything except "most are cheap"; a hundred bins expose the spiky gaps around round price points. Neither is wrong, but neither is neutral either — the bin width is an assumption you are making on the reader's behalf.

`plt.subplots` is how you put several panels side by side: it hands back a figure and an array of axes, and each seaborn call takes `ax=` to say which panel to draw on.

**Exercise 1.** Using `orders`, generate a histogram of the `profit` column.


In [ ]:
# Your code here

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

In [ ]:
# Same data, axis limited to the bulk of it, with a reference line at break-even.
sns.histplot(data=orders, x="profit", bins=60)
plt.xlim(-500, 500)
plt.axvline(0, color="red", linestyle="--")
plt.title("Order profit, zoomed to -500 to +500")
plt.xlabel("Profit (USD)")
plt.show()


> `plt.xlim` zooms the view; it does **not** filter the data, and the bars outside the window are still there, just off-screen. Say so in the caption when you do this — a reader who assumes they are seeing everything will misread the chart.


### Boxplots

A boxplot compresses a distribution into five numbers — minimum, first quartile, median, third quartile, maximum — and draws anything beyond 1.5 times the interquartile range as an individual point. That makes it the fastest way to compare many groups at once, and a good outlier detector.


In [ ]:
# Two row filters combined with &, then one box per team.
al_east = mlb.loc[(mlb.lgID == 'AL') & (mlb.divID == 'E'), :]

plt.figure(figsize=(8, 4))
sns.boxplot(data=al_east, x="teamID", y="W")
plt.title("Wins per season, AL East teams (all seasons on record)")
plt.xlabel("Team")
plt.ylabel("Wins")
plt.show()


Each box covers the middle half of that team's seasons, the line inside is the median, and the isolated dots are unusually good or bad years.

Note the trap in the filter: this uses `mlb`, not the 2021-2023 subset, so every AL East season since the division was created in 1969 is in one picture. Baltimore, Boston and the Yankees contribute 55 seasons each; `WS2` contributes **three**. A box drawn from three numbers is not comparable to a box drawn from fifty-five, and the plot gives you no hint of that — always check the group sizes. The pandemic-shortened 2020 season (60 games) sits low for everybody, for reasons that have nothing to do with the teams.

Each parenthesised condition is required, by the way: `&` binds tighter than `==` in Python, so `mlb.lgID == 'AL' & mlb.divID == 'E'` raises an error.

In [ ]:
# Boxplots are also an outlier detector. The survey file loaded at the top has two planted problems.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

sns.boxplot(data=survey, x="Age", ax=axes[0])
axes[0].set_title("Respondent age")

sns.boxplot(data=survey, x="Income", ax=axes[1])
axes[1].set_title("Respondent income")

plt.tight_layout()
plt.show()


In [ ]:
# Confirm what the boxplots flagged.
survey[['Age', 'Income']].describe().round(1)


Two impossible values jump straight out: an age of 148 and an income of -68,000. Neither is a subtle statistical outlier — they are data-entry errors, and both would quietly distort any mean, correlation or chart built on this file.

This is the argument for plotting distributions **before** analysing anything. `.describe()` would also have shown you a maximum age of 148, but only if you thought to look; the boxplot puts it in front of you whether you were looking or not.

# Follow up Questions and Exercises — Part 2: an analyst task

**Exercise 1.** Support satisfaction by channel and plan.

Your manager believes **Pro users get faster responses and higher satisfaction**,  
and that **chat support may outperform other channels**.

Is this true?

### Tips
1. Review the dataset — what looks inconsistent or messy?  
2. Consider standardizing values and ensuring the that values are consistent!

In [ ]:
# The starter data, typed in as a list of dictionaries -- one dictionary per ticket.
data = [
    {"ticket_id": "T001", "plan": "Pro",   "channel": "Email",   "response_time_min": 45,  "satisfaction": 5, "country": "US"},
    {"ticket_id": "T002", "plan": "PR0",   "channel": " chat ",  "response_time_min": 15,  "satisfaction": 4, "country": "Us "},
    {"ticket_id": "T003", "plan": "Basic", "channel": "Phone ",  "response_time_min": 60,  "satisfaction": 3, "country": "CA"},
    {"ticket_id": "T004", "plan": "basic", "channel": "email",   "response_time_min": 40,  "satisfaction": 2, "country": "ca"},
    {"ticket_id": "T005", "plan": "Pro ",  "channel": "Chat",    "response_time_min": 10,  "satisfaction": 5, "country": "US "},
    {"ticket_id": "T006", "plan": "BASIC ", "channel": "phone",  "response_time_min": 80,  "satisfaction": 3, "country": "CA "},
    {"ticket_id": "T007", "plan": "pro",   "channel": "Email ",  "response_time_min": 25,  "satisfaction": 4, "country": "US"},
    {"ticket_id": "T008", "plan": "Basic", "channel": "CHAT",    "response_time_min": 30,  "satisfaction": 5, "country": "us"},
    {"ticket_id": "T009", "plan": "PR0",   "channel": "Phone",   "response_time_min": 90,  "satisfaction": 2, "country": "CA"},
    {"ticket_id": "T010", "plan": "Basic", "channel": "chat ",   "response_time_min": 25,  "satisfaction": None, "country": "CA"},
]

support = pd.DataFrame(data)
support


In [ ]:
# Your code here -- step 1: count the plans and channels before cleaning

In [ ]:
# Your code here -- step 2: standardise the plan, channel and country values

In [ ]:
# Your code here -- step 3: average response time, average satisfaction and ticket count per plan and channel

In [ ]:
# Your code here -- step 4: chart the comparison

In [ ]:
# Your code here -- step 5 (optional): a second chart that shows both metrics at once

**Answer — is the manager right?**

*Provide your answer here*

# Follow up Questions and Exercises — Part 3: after class

Work through these after class.

**Question 1.** You want to show the average sales for each of the four customer segments. Should that be a bar chart or a line chart, and why?


In [ ]:
# Your code here (optional): draw it both ways and compare

**Answer**

*Provide your answer here*

**Question 2.** A colleague sends you a correlation matrix showing that `unit_price` and `sales` correlate at 0.43, and concludes that raising unit prices will raise sales. What would you say?


In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 1.** Which product sub-categories lose money? Build the table, then chart it so the losers are obvious at a glance.


In [ ]:
# Your code here

**Answer**

*Provide your answer here*